# Alkahest — Library Tour

A hands-on walkthrough of the major features in **Alkahest**, a high-performance computer algebra
system for Python.  Each section is self-contained; run them in order after installing the package.

**Stack:** Rust kernel → FLINT/Arb → egglog → MLIR/LLVM → PyO3 → Python

---

### Installation

The standard PyPI wheel works for everything in this notebook **except** the JIT and Gröbner
sections, which require the optional build features described below.

```
# Standard install (no LLVM, no Gröbner solver)
pip install alkahest

# Full source build with all optional features
#   pip install maturin
#   maturin develop --release --features "parallel egraph jit groebner"
```

> **JIT** (`compile_expr`, `jit` decorator) requires `--features jit` and LLVM 15 at build time.
> **Gröbner / solve / Diophantine / homotopy** require `--features groebner`.

In [ ]:
%pip install -q alkahest

In [ ]:
import alkahest as ak
print("alkahest version:", ak.__version__)
print("JIT available:   ", ak.jit_is_available())

---
## 1  ExprPool and symbols

All symbolic expressions live in an **`ExprPool`** — a hash-consed DAG that guarantees structural
sharing and O(1) equality checks.  Symbols, integers, and every compound expression are nodes in
this pool.

In [ ]:
pool = ak.ExprPool()
x = pool.symbol("x")
y = pool.symbol("y")
n = pool.symbol("n")

# Integer and rational literals
two = pool.integer(2)
neg1 = pool.integer(-1)

# Compound expressions
expr = x**2 + two * x + pool.integer(1)
print("x² + 2x + 1 =", expr)

# Hash-consing: identical expressions share the same node
a = x + pool.integer(0)
b = x + pool.integer(0)
print("Same node?   ", a == b)   # True

---
## 2  Simplification

`simplify` rewrites an expression using an e-graph (egglog) backend when available, otherwise a
rule-based simplifier.  Every call returns a **`DerivedResult`** with `.value` (the simplified
expression) and `.steps` (the derivation log).

In [ ]:
r = ak.simplify(x + pool.integer(0))
print("x + 0          →", r.value)
print("steps          :", len(r.steps))

r2 = ak.simplify((x + pool.integer(0)) * pool.integer(1))
print("(x + 0) * 1    →", r2.value)

# Trig identity
r3 = ak.simplify(ak.sin(x)**2 + ak.cos(x)**2)
print("sin²x + cos²x  →", r3.value)

# Inspect the first few rewrite steps
print("\nDerivation log for (x+0)*1:")
for step in r2.steps[:4]:
    print(f"  {step['rule']:30s}  {step['before']}  →  {step['after']}")

---
## 3  Differentiation

In [ ]:
# Basic rules
print("d/dx x³          =", ak.diff(x**3, x).value)
print("d/dx sin(x)      =", ak.diff(ak.sin(x), x).value)
print("d/dx exp(x)      =", ak.diff(ak.exp(x), x).value)
print("d/dx log(x)      =", ak.diff(ak.log(x), x).value)

# Chain rule
d_sin_x2 = ak.diff(ak.sin(x**2), x)
print("d/dx sin(x²)     =", d_sin_x2.value)
print("  (expect 2*x*cos(x²))")

# Product rule
d_prod = ak.diff(x * ak.exp(x), x)
print("d/dx x·exp(x)    =", d_prod.value)

# Second derivative
d2 = ak.diff(ak.diff(ak.sin(x), x).value, x)
print("d²/dx² sin(x)    =", ak.simplify(d2.value).value)

---
## 4  Integration

In [ ]:
print("∫ x² dx          =", ak.integrate(x**2, x).value)
print("∫ sin(x) dx      =", ak.integrate(ak.sin(x), x).value)
print("∫ cos(x) dx      =", ak.integrate(ak.cos(x), x).value)
print("∫ exp(x) dx      =", ak.integrate(ak.exp(x), x).value)
print("∫ 1/x dx         =", ak.integrate(x**-1, x).value)

# Verify ∂/∂x ∫f = f
f = x**3 + pool.integer(2)*x
antideriv = ak.integrate(f, x).value
roundtrip = ak.simplify(ak.diff(antideriv, x).value).value
print("\nd/dx ∫(x³+2x)dx  =", roundtrip)

---
## 5  Truncated series

In [ ]:
# cos(x) around 0, 6 terms
s_cos = ak.series(ak.cos(x), x, pool.integer(0), 6)
print("cos(x) series:", s_cos.expr)

# sin(x) around 0, 5 terms
s_sin = ak.series(ak.sin(x), x, pool.integer(0), 5)
print("sin(x) series:", s_sin.expr)

# Laurent series: 1/x around 0
s_inv = ak.series(x**(-1), x, pool.integer(0), 4)
print("1/x   series: ", s_inv.expr)

---
## 6  Polynomial representations

Alkahest exposes **three explicit polynomial types** backed by FLINT.  Conversion is always opt-in
— no silent performance cliffs.

In [ ]:
# ── UniPoly: dense univariate ─────────────────────────────────────────────
p = ak.UniPoly.from_symbolic(x**3 + pool.integer(-2)*x + pool.integer(1), x)
print("UniPoly p     =", p)
print("degree        =", p.degree())
print("coefficients  =", p.coefficients())   # constant-first

# GCD
a_poly = ak.UniPoly.from_symbolic(x**2 + pool.integer(-1), x)  # x²-1
b_poly = ak.UniPoly.from_symbolic(x   + pool.integer(-1), x)   # x-1
print("gcd(x²-1,x-1) =", a_poly.gcd(b_poly))

# Factorization over ℤ
fac = a_poly.factor_z()
print("factor(x²-1)  :", [(str(f), e) for f, e in fac.factor_list()])

# ── MultiPoly: sparse multivariate ───────────────────────────────────────
print()
mp = ak.MultiPoly.from_symbolic(x**2 * y + x * y**2, [x, y])
print("MultiPoly     =", mp)
print("total_degree  =", mp.total_degree())

# ── RationalFunction: automatic GCD normalization ─────────────────────────
print()
rf = ak.RationalFunction.from_symbolic(
    x**2 + pool.integer(-1),   # numerator
    x   + pool.integer(-1),    # denominator
    [x]
)
print("(x²-1)/(x-1) =", rf)   # → x + 1

---
## 7  Interval (ball) arithmetic

`ArbBall` wraps Arb's `arb_t` — a real interval `[mid ± rad]` at arbitrary MPFR precision.
`interval_eval` propagates enclosures through a symbolic expression, giving a **rigorous** bound.

In [ ]:
import math

# Basic ball arithmetic
a_ball = ak.ArbBall(2.0, 0.5)   # [1.5, 2.5]
b_ball = ak.ArbBall(3.0, 0.5)   # [2.5, 3.5]
print("a =", a_ball, "  lo/hi:", round(a_ball.lo, 2), round(a_ball.hi, 2))
print("b =", b_ball)
print("a + b =", a_ball + b_ball, "  contains 5.0:", (a_ball + b_ball).contains(5.0))
print("a * b =", a_ball * b_ball, "  contains 6.0:", (a_ball * b_ball).contains(6.0))

# Transcendental evaluation
pi2 = ak.ArbBall(math.pi / 2, 0.01)
s = pi2.sin()
print("\nsin(π/2 ± 0.01) =", s, "  contains 1.0:", s.contains(1.0))

# interval_eval on a symbolic expression: f(x) = x² + 1, x ∈ [2.9, 3.1]
f_expr = x**2 + pool.integer(1)
x_ball = ak.ArbBall(3.0, 0.1)
result = ak.interval_eval(f_expr, {x: x_ball})
print(f"\nx²+1 at x∈[2.9,3.1]:")
print(f"  enclosure = {result}")
print(f"  lo={result.lo:.4f}  hi={result.hi:.4f}")
print(f"  contains 10.0: {result.contains(10.0)}")

# Higher precision → tighter bound
print("\nRadius of exp([−0.001,0.001]) at various precisions:")
for prec in [32, 64, 128, 256]:
    z = ak.ArbBall(0.0, 1e-3, prec)
    print(f"  prec={prec:3d}  rad={z.exp().rad:.3e}")

---
## 8  Non-commutative algebra — Pauli matrices

Symbols can opt out of multiplicative commutativity with `commutative=False`.
`simplify_pauli` applies the built-in Pauli rewrite table (σₓσᵧ = iσ_z, etc.).

In [ ]:
sx = pool.symbol("sx", "complex", commutative=False)
sy = pool.symbol("sy", "complex", commutative=False)
sz = pool.symbol("sz", "complex", commutative=False)

# Non-commutativity: σₓσᵧ ≠ σᵧσₓ
xy = sx * sy
yx = sy * sx
print("sx*sy != sy*sx:", xy != yx)

# Pauli product rule: σₓσᵧ = i·σz
r = ak.simplify_pauli(xy)
print("simplify_pauli(sx*sy) =", r.value)

# Pauli squares: σₓ² = I
r2 = ak.simplify_pauli(sx * sx)
print("simplify_pauli(sx*sx) =", r2.value)

# Clifford orthogonal pair
e1 = pool.symbol("cliff_e1", commutative=False)
e2 = pool.symbol("cliff_e2", commutative=False)
r3 = ak.simplify_clifford_orthogonal(e1 * e2)
print("cliff_e1 * cliff_e2   =", r3.value)
r4 = ak.simplify_clifford_orthogonal(e1 * e1)
print("cliff_e1 * cliff_e1   =", r4.value)

---
## 9  Output formatting — LaTeX and Unicode

In [ ]:
expr_fmt = ak.sin(x)**2 + ak.cos(x)**2
print("LaTeX  :", ak.latex(expr_fmt))
print("Unicode:", ak.unicode_str(expr_fmt))

# Parse from string
e = ak.parse("x^2 + 2*x + 1", pool, {"x": x})
print("Parsed :", e)
print("LaTeX  :", ak.latex(e))

---
## 10  Symbolic summation and products

In [ ]:
k = pool.symbol("k")

# Indefinite sum: Σ k
s_k = ak.sum_indefinite(k, k)
print("sum_indefinite(k, k)  =", s_k.value)

# Definite sum: Σ_{k=0}^{n} k
s_def = ak.sum_definite(k, k, pool.integer(0), n)
print("sum_definite(k,k,0,n) =", s_def.value)

# Gamma-involving term
term = ak.simplify(k * ak.gamma(k + pool.integer(1))).value
print("sum_indefinite(k·Γ(k+1)) =", ak.sum_indefinite(term, k).value)

# Definite product: ∏_{k=1}^{n} k  (= n!)
P = ak.Product(k, (k, pool.integer(1), n))
print("\n∏_{k=1}^{n} k =", ak.simplify(P.doit().value).value)

---
## 11  Number theory

In [ ]:
import alkahest.number_theory as nt

# Primality
print("isprime(2¹²⁷-1)  =", nt.isprime(2**127 - 1))
print("isprime(2¹²⁸-1)  =", nt.isprime(2**128 - 1))

# Factorization
print("factorint(2³²-1) =", nt.factorint(2**32 - 1))

# Discrete log: log_3(13) mod 17 == 4  (3^4 = 81 ≡ 13 mod 17)
print("discrete_log(13,3,17) =", nt.discrete_log(13, 3, 17))

# Modular square root
r_sq = nt.nthroot_mod(144, 2, 401)
print(f"nthroot_mod(144,2,401) = {r_sq}, verify: {pow(r_sq, 2, 401)} == {144 % 401}")

# Next prime
print("nextprime(100) =", nt.nextprime(100))

# Euler totient
print("totient(36)    =", nt.totient(36))

---
## 12  Recurrences (`rsolve`)

In [ ]:
f = lambda *a: pool.func("f", list(a))

# f(n) - f(n-1) = 1  →  f(n) = n + C₀
eq = ak.simplify(f(n) - f(n + pool.integer(-1)) - pool.integer(1)).value
print("f(n)-f(n-1)-1=0  →", ak.rsolve(eq, n, "f", None))

# Fibonacci: f(n) - f(n-1) - f(n-2) = 0, f(0)=0, f(1)=1
fib_eq = ak.simplify(
    f(n) - f(n + pool.integer(-1)) - f(n + pool.integer(-2))
).value
print("Fibonacci        →", ak.rsolve(fib_eq, n, "f", {0: pool.integer(0), 1: pool.integer(1)}))

---
## 13  Composable transformations — `trace` / `grad`

In [ ]:
@ak.trace
def h(x):
    return ak.sin(x**2)

dh = ak.grad(h)          # symbolic gradient
print("h(x)  = sin(x²)")
print("h'(x) =", dh(x))

---
## 14  JIT compilation  *(requires `--features jit` + LLVM 15)*

If you installed the standard PyPI wheel, `jit_is_available()` will be `False` and the cell below
will skip.  Build from source with `--features jit` to enable.

In [ ]:
if ak.jit_is_available():
    f_jit = ak.compile_expr(x**2 + pool.integer(1), [x])
    print("f(3.0) =", f_jit([3.0]))   # → 10.0

    @ak.trace
    def g(x):
        return ak.sin(x**2)

    g_fast = ak.jit(ak.grad(g))
    print("g'(1.0) ≈", g_fast([1.0]))
else:
    print("JIT not available — build with: maturin develop --release --features jit")

---
## 15  Polynomial solving and Diophantine  *(requires `--features groebner`)*

In [ ]:
try:
    # Triangular decomposition (regular chains)
    eq1 = x**2 + y**2 - pool.integer(1)
    eq2 = y - x
    chains = ak.triangularize([eq1, eq2], [x, y])
    print("triangularize(x²+y²=1, y=x):")
    for c in chains:
        print(" ", c)

    # Linear Diophantine: 3x + 5y = 1
    sol = ak.diophantine(
        pool.integer(3)*x + pool.integer(5)*y - pool.integer(1), [x, y]
    )
    print("\n3x + 5y = 1:", sol.kind, sol)

    # Pell equation: x² - 2y² = 1
    pell = ak.diophantine(x**2 - pool.integer(2)*y**2 - pool.integer(1), [x, y])
    print("x² - 2y² = 1:", pell.kind, "fundamental =", pell.fundamental)

except Exception as exc:
    print("Groebner feature unavailable:", exc)
    print("Build with: maturin develop --release --features groebner")

---
## 16  Homotopy continuation  *(requires `--features groebner`)*

In [ ]:
try:
    sols = ak.solve([x**2 + neg1, y**2 + neg1], [x, y], method="homotopy")
    print(f"x²=1, y²=1 — {len(sols)} solutions:")
    for s in sols:
        print(" ", {str(k): round(float(str(v)), 4) for k, v in s.items()})

    cs = ak.solve_numerical([x**2 + neg1], [x])
    for c in cs:
        print(f"  coord={c.coordinates}  certified={c.smale_certified}")

except Exception as exc:
    print("Groebner/homotopy feature unavailable:", exc)
    print("Build with: maturin develop --release --features groebner")

---
## 17  Derivation log and Lean 4 certificates

Every top-level operation returns a `DerivedResult`.  When a Lean 4 certificate is available,
`.certificate` holds the proof term.

In [ ]:
result = ak.diff(x**3, x)
print("value      :", result.value)
print("steps      :", len(result.steps))
print("certificate:", result.certificate)   # None unless Lean backend is built

print("\nFirst rewrite steps:")
for step in result.steps[:6]:
    print(f"  {step['rule']:30s}  {step['before']}  →  {step['after']}")